## Dữ liệu sau viết lại
Summary có căn cứ trong CV gốc, bullet chỉnh diễn đạt bằng quy tắc. Chọn kernel kltn.


In [ ]:
import json
from pathlib import Path
from datasets import Dataset, load_dataset

# Tìm thư mục gốc khi notebook được mở từ các thư mục khác nhau.
project_root = None
for folder in [Path.cwd(), *Path.cwd().parents]:
    build_file = folder / 'ai/training/data_processing/build_data.py'
    if build_file.exists():
        project_root = folder
        break

if project_root is None:
    raise FileNotFoundError('Không tìm thấy thư mục gốc KLTN')

processed_dir = project_root / 'ai/data/processed'
dataset_path = processed_dir / 'rewritten_dataset.json'
processed_records = json.loads(dataset_path.read_text(encoding='utf-8'))
dataset = Dataset.from_list(processed_records)
print('Số mẫu:', len(dataset))
dataset[0]


In [ ]:
# Xem riêng các phần của một mẫu.
index = 0
dataset[index]['target_cv']


In [ ]:
changes_path = processed_dir / 'rewrite_changes.json'
rewrite_changes = json.loads(changes_path.read_text(encoding='utf-8'))
sample_id = dataset[index]['sample_id']
sample_change = None
for change in rewrite_changes:
    if change['sample_id'] == sample_id:
        sample_change = change
        break

if sample_change is None:
    raise ValueError(f'Không tìm thấy thay đổi của mẫu {sample_id}')

print(json.dumps(sample_change, ensure_ascii=False, indent=2))


In [ ]:
print((processed_dir / 'rewrite_report.json').read_text())
split_dir = PROJECT_ROOT / 'ai/data/splits'
data_files = {
    'train': str(split_dir / 'train.json'),
    'validation': str(split_dir / 'validation.json'),
    'test': str(split_dir / 'test.json'),
}
training_data = load_dataset('json', data_files=data_files)
training_data['train'][0]


## Tái tạo dữ liệu (chạy khi cần)
Cell sau chạy toàn bộ pipeline và dọn các file trung gian cũ.


In [ ]:
import subprocess
import sys
command = [sys.executable, '-m', 'ai.training.data_processing.build_data']
subprocess.run(command, cwd=project_root, check=True)
